# Activity: Python Data Science with Pandas

**Initial Due Date: 2026-01-07 10:00AM**  
**Final Due Date: 2026-01-12 4:15PM**

## Learning Objectives

By the end of this notebook, you will be able to:

1.  Create and manipulate Pandas DataFrames
2.  Apply vectorized selection and mutation operations
3.  Apply split-apply-combine to transform data

> #### ❗ It is OK add cells (just don’t change specified function/variable names)
>
> It is OK to add cells as you work through each activity, and we
> encourage you to leave those in place to show your thought process.
> But **do not** modify or “shadow” any specified function or variable
> names, as that will be break the automated tests.

## Introduction

Decision trees are a supervised learning method used for classification
and regression. They represent decisions as a tree of nodes: internal
nodes test feature values, branches correspond to outcomes of those
tests, and leaves give predicted labels or values.

The following figure shows an example decision tree for deciding whether
to wait for a table at a restaurant based the type of restaurant, day of
the week, busyness, etc. To make a prediction for a new observation we
start at the root and apply the conditions for that observation until we
reach a a decision at a leaf node. For example, if “Patrons” is “Full”,
“Hungry” is “Yes” and “Type” is “French” we would predict “Yes” (to
wait). Convince yourself of that prediction by tracing the corresponding
path through the tree.

<figure>
<img
src="https://middcs.github.io/csci-1010-w26/assets/img/figs/decision-tree.png"
alt="Example decision tree for deciding when to wait for a table at a restaurant. Image credit: @russellArtificialIntelligence2020" />
<figcaption aria-hidden="true">Example decision tree for deciding when
to wait for a table at a restaurant. Image credit: <span
class="citation"
data-cites="russellArtificialIntelligence2020">@russellArtificialIntelligence2020</span></figcaption>
</figure>

We can learn decisions trees from labeled data, like shown below. We
have two parts, the input data, which describes the restaurant, day of
the week, etc., and the *labels* or outcomes, which indicate whether the
person decided to wait or not. Following community conventions, we will
use $X$ to denote the input data and $y$ to denote the labels. Each row
corresponds to one observation. The columns of the input data are
*features* or *input attributes*. The ultimate goal is to predict the
label for new observations, i.e., new combinations of the input
attributes, we have not seen before.

<figure>
<img
src="https://middcs.github.io/csci-1010-w26/assets/img/figs/decision-tree-data.png"
alt="Restaurant waiting data for learning a decision tree. Image credit: @russellArtificialIntelligence2020" />
<figcaption aria-hidden="true">Restaurant waiting data for learning a
decision tree. Image credit: <span class="citation"
data-cites="russellArtificialIntelligence2020">@russellArtificialIntelligence2020</span></figcaption>
</figure>

Here will focus on a binary decisions, i.e., those with true/false
outcomes or labels. One algorithm for learning the structure of these
kind of trees from a given labeled data set is to recursively pick as
the next branch the attribute that “best” separates the remaining
positive and negative examples. “Best” is doing a lot of work in that
sentence and needs more specificity. Here we will select the attribute
with the maximum *information gain*.

> #### 🤔 We will gloss over most of the details for decision trees
>
> For now we will gloss over most of the details of decision trees to
> focus on the use of Pandas. Check out
> [CSCI311](https://catalog.middlebury.edu/courses/view/course-CSCI0311)
> to learn more about this very cool machine learning tool!

We define the information gain $IG$ for some data $X$, labels $y$, and
attribute $A$ as

$$
H(V) = -\sum_{k} p(v_k) \log_2 p(v_k)
$$

<span id="eq-information-gain">$$
\mathrm{IG}(X, y, A) = H(y) - \sum_{v \in \mathrm{Vals}(A)} \frac{|X_v|}{|X|} H(y_v)
 \qquad(1)$$</span>

where:

-   $H(V)$ is the entropy of a random variable with values $v_k$
    observed with probability $p(v_k)$.
-   $X_v$ and $y_v$ are the subsets of $X$ and $y$ where attribute $A$
    has value $v$.
-   $|X_v|$ and $|X|$ are the number of observations in $X_v$ and $X$,
    respectively (i.e., the number of rows).
-   $\mathrm{Vals}(A)$ is the set of possible values for attribute $A$.

Entropy is a measure of uncertainty (the unit is bits). We could
calculate the entropy for the example labels above (wait or not wait) as
shown below. In this context, there are two possible values, “yes” and
“no”, each observed with probability $\frac{6}{12}$. Since both outcomes
are equally likely, the uncertainty is maximized.

$$
\begin{split}
H(y) &= -\left(p_{yes} \log_2 p_{yes} + p_{no} \log_2 p_{no}\right) \\
&= -\left(\frac{6}{12} \log_2 \frac{6}{12} + \frac{6}{12} \log_2 \frac{6}{12}\right) \\
&= 1 \text{ bit}
\end{split}
$$

The information gain is the difference between the entropy of the parent
node in the tree and weighted average of entropy of the children node
when splitting on a particular attribute, i.e., how much that choice of
attribute reduces the uncertainty. The best choice of attribute is the
one the reduces the uncertainty the most.

> #### 😱 Oh no!!! $\log_2 0$!!!
>
> You may be wondering what happens when $p(v_k) = 0$? After all,
> $\log_2 0$ is undefined (or sometimes by convention defined to be
> $-\infty$). In many information theory applications, we stipulate that
> $0 \log 0 = 0$. This agrees with how Numpy and Pandas behave: they
> return `nan` for $\log 0$ but multiplying by 0 results in 0. Using the
> `observed` argument of `groupby` (see below) can also help avoid this
> issue.

Get started with the necessary imports:

In [ ]:
import numpy as np
import pandas as pd

## Part A: Compute the entropy

### Exercise A1

Write a function `entropy` that uses Pandas and NumPy to compute the
entropy for a `Series` of labels, `y`. Although we will primarily use
this for binary outcomes, your function should work for labels of any
cardinality. Recall that you can mix Pandas and NumPy, i.e., invoke
NumPy functions with Pandas series/columns as the argument.

Keeping with our focus for today, think about how you could use
split-apply-combine. Note that `groupby` with `Series` can be more
awkward than the examples we saw with `DataFrames`. We still need to
specify what to group on. Sometimes that is the index but often it is
the values in the series. For the latter, we provide the `Series` as
both the receiver and the argument to `groupby`, e.g.,
`vals.groupby(vals)`.

> #### 💡 `groupby` tips: `observed`
>
> If the grouping variable is categorical, i.e., has a fixed set of
> possible values, the optional argument `observed=True` will only
> report values for groups that exist, i.e., it will skip groups with no
> observations. While `observed=False` will report values for all
> groups, even those without any observations. The default of Pandas is
> in flux, so we encourage you to make the explicit choice relevant to
> your situation. In this case, we don’t need or want to consider groups
> with no observations (since they do not contribute to the entropy and
> may lead to the $\log_2 0$ issue noted above).

In [ ]:
def entropy(y: pd.Series) -> float:
    """Return entropy for labels pandas Series"""
    # TODO: Your code here
    return 0.0

To test your function, let’s start with the extremes. All identical
values have zero uncertainty, i.e., an entropy of zero,

> #### ❓ You may see a “negative” 0
>
> You may see a “negative” 0, i.e., -0.0. That is not an error. It is an
> artifact of how floating point values separately represent the sign
> and the value and thus have two “zeros”.

In [ ]:
entropy(pd.Series([0, 0, 0, 0, 0]))

while values that are evenly distributed have the maximum uncertainty.
Entropy measures the number of bits theoretically required to losslessly
compress data. If we have two equally likely values we will need 1 bit,
4 equally likely values will need 2 bits, and so on.

In [ ]:
entropy(pd.Series([0, 0, 0, 1, 1, 1]))

Your function should work for any number of values, not just the binary
examples above. If doing so requires special cases in your code, we
encourage you revisit your implementation to use an approach (hint,
split-apply-combine) that work for any number of values.

In [ ]:
entropy(pd.Series([0, 0, 0, 1, 1, 1, 2]))

## Part B: Compute the information gain

### Exercise B1

Write a function `information_gain` that takes 3 arguments: `X`, a
DataFrame with the observations, `y`, a Series with the corresponding
labels, and `attr`, a string, and returns the information gain for
splitting the data on that attribute, i.e., your function should
implement the information gain equation above
<a href="#eq-information-gain" class="quarto-xref">Equation 1</a>. You
are welcome and encouraged to add any other functions you need to make
your implementation more readable and maintainable.

***Note***: *While any reasonable implementation will be accepted, it is
possible to complete the function in fewer than 5 lines of code,
including the return statement. You just need to trust your Pandas and
NumPy!*

Some suggestions:

1.  Consider how to use split-apply-combine. In addition to the
    aggregations we saw in class, you can use `apply` to execute a
    function on each group (provided as an argument to that function).
    Recall that you can create anonymous functions at the point you need
    them using [`lambda`
    expressions](https://docs.python.org/3/tutorial/controlflow.html#lambda-expressions).
2.  The `len` function returns the number of rows in a `DataFrame`.
3.  You can use the indices (row labels) of one `DataFrame`/`Series`,
    available via the `.index` attribute, to select data in another
    `DataFrame`/`Series`. Specifically, each row in `X` and its
    corresponding label in `y` are linked by their shared labels
    (initialized to 0, 1, … when we first create those data structures).
    Those labels are preserved through grouping and other operations
    that otherwise change the order of the rows, thus for some group `g`
    in `X`, the corresponding labels are `y[g.index]`.

In [ ]:
def information_gain(X: pd.DataFrame, y: pd.Series, attr: str) -> float:
    """Return the expected reduction in entropy from splitting X,y by attr"""
    # TODO: Implement information gain metric for selecting attributes
    # TODO: Your code here
    return 0.0

To test your implementation you can use the following data, which
describes when someone would play tennis depending on the weather. Note
the `pd.Categorical` columns. This creates “categorical” data, i.e., a
type with a discrete set of values (the same as a `factor` in R, and
analogous to “enum” types in Python et al.) Categorical types are a key
tool for “marking” data as discrete (which informs downstream analyses)
and recording the possible values.

In [ ]:
tennis_X = pd.DataFrame({
    "Outlook": pd.Categorical(["Sunny", "Sunny", "Overcast", "Rain", "Rain", "Rain", "Overcast", "Sunny", "Sunny", "Rain", "Sunny", "Overcast", "Overcast", "Rain", "Rain", "Rain"]),
    "Temperature": pd.Categorical(["Hot", "Hot", "Hot", "Mild", "Cool", "Cool", "Cool", "Mild","Cool", "Mild", "Mild", "Mild", "Hot", "Mild", "Mild", "Mild"]),
    "Humidity": pd.Categorical(["High", "High", "High", "High", "Normal", "Normal", "Normal", "High", "Normal", "Normal", "Normal", "High", "Normal", "High", "High", "High"]),
    "Wind": pd.Categorical(["Weak", "Strong", "Weak", "Weak", "Weak", "Strong", "Strong", "Weak","Weak", "Weak", "Strong", "Strong", "Weak", "Strong", "Weak", "Weak"]),
})

tennis_y = pd.Series([0, 0, 1, 1, 1, 0, 1, 0, 1, 1, 1, 1, 1, 0, 0, 0])

tennis_X, tennis_y

Convince yourself the expected information gains are \[0.254, 0.036,
0.213, .007\]:

In [ ]:
for attr in ["Outlook", "Temperature", "Humidity", "Wind"]:
    print(information_gain(tennis_X, tennis_y, attr))

## Part C: Select the best attribute

### Exercise C1

Write a function `find_best_split` that returns the column name for the
best attribute to pick to split the data, i.e., the column with the
largest information gain. In the case of a tie we want to randomly
select among the possible best attributes.

It is possible for all attributes to have an information gain of 0. Make
sure that your method for selecting the maximum gain will identify an
attribute even if they all have a gain of 0.

Some suggestions:

1.  The column labels are available as a sequence of strings via the
    `.columns` attribute
2.  If you otherwise deterministically break ties when computing the
    maximum, e.g., always select the first like Python’s built-in `max`
    function, you can effect random tie breaking by randomly permuting
    the attribute order before picking the maximum, e.g. with
    [`np.random.permutation`](https://numpy.org/doc/stable/reference/random/generated/numpy.random.permutation.html).
    Unlike the in-place shuffle, `permutation` creates a copy of the
    argument sequence and so can be used with the immutable `.columns`
    attribute.
3.  There are many ways to determine the maximum (e.g., with a function,
    with a loop). Any reasonable approach will be accepted. However, in
    keeping with our goal to eliminate explicit loops, investigate the
    `key` argument of the Python [`max`
    function](https://docs.python.org/3/library/functions.html#max). If
    you provide a function as the `key` argument, `max` will use the
    values produced by that function to compare the items instead of the
    original values (but still return the original value). For example
    `max(values, key=lambda x: -x)` actually returns the minimum value.
    In this context, you could use the `key` function to compute the
    information gain for each attribute when comparing them.

In [ ]:
def find_best_split(X: pd.DataFrame, y: pd.Series) -> str:
    """Return the name of the column in X with maximum information gain"""
    # TODO: Your code here
    return None

To test your implementation by compute the best split for the tennis
data. From our results above, we know “Outlook” should be the best
split.

In [ ]:
find_best_split(tennis_X, tennis_y)

> #### 💡 Customizing min, max, sort, etc.
>
> We observe that many programming languages, like Python and R, have
> similar semantics, i.e., we can expect certain common features across
> languages (albeit often with different syntax). One such feature is
> the ability to customize how the underlying comparison for minimum,
> maximum, and sorting is performed. In Python, this is done via the
> `key` argument of functions like `min`, `max`, and `sorted`. In R,
> this is done via the `order` function combined with indexing.
> Understanding these commonalities can help you transfer your knowledge
> between languages.

## Collaboration statement

In a markdown cell below, briefly list who or what you collaborated with
and how. Cite any sources here or with relevant inline comments in your
code. Acknowledge all contributors, both people and AI, and what
portions of this notebook they contributed. You do not need to cite or
acknowledge any material provided in the starter file(s).

## Submitting your notebook

You will simultaneously submit the following two files to the relevant
assignment on [Gradescope](https://gradescope.com) via the “Upload
option” (guide
[here](https://guides.gradescope.com/hc/en-us/articles/21865616724749-Submitting-a-Code-assignment)).
**Both files must be uploaded at the same time and the file names must
match the specification exactly for the autotesting to run
successfully.**

1.  `activity_python_pandas.ipynb`: Your completed IPython notebook. You
    can obtain this via the “File→Download→Download .ipynb” menu option
    in Colab.
2.  `activity_python_pandas.py`: Your completed IPython notebook as a
    Python file. You can obtain this via the “File→Download→Download
    .py” menu option in Colab. This file is used to provide line-level
    feedback on your submission.

You can submit multiple times, with only the most recent submission
(before the final due date) assessed for credit. Gradescope will run a
series of automated unit tests on your notebook (which may takes 10s of
seconds depending on the complexity of the notebook). Note that the
tests performed by Gradescope are limited. Passing all of the visible
tests does not guarantee that your submission correctly satisfies all of
the requirements of the assignment.